In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
# import sys
# import os

# home = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
# print(home)

catalog = dbutils.widgets.get("catalog")
checkpoints_dir = dbutils.widgets.get("checkpoints_dir")

schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("region_id", IntegerType(), True),
    StructField("region_name", StringType(), True),
    StructField("operation", StringType(), True),
    StructField("event_time", TimestampType(), True)
])

df = (spark.readStream
        .table(f"{catalog}.brz.regions_raw"))

df = df.select("value")

parsed_df = (df.withColumn("parsed", F.from_json(F.col("value"), schema))
                .select("parsed"))

parsed_df = (parsed_df.select("parsed.region_id", F.initcap(F.col("parsed.region_name")).alias("city"),
                                F.current_timestamp().alias("processed_time")))

query = (parsed_df.writeStream
            .format("delta")
            .option("checkpointLocation", f"abfss://checkpoints@jayveeradlsdevtest.dfs.core.windows.net/{checkpoints_dir}/brz_checkpoints/regions_checkpoints")
            .trigger(availableNow = True)
            .outputMode("append")
            .table(f"{catalog}.slv.regions"))

query.awaitTermination()

c:\Users\hkand\streaming\sales_streaming_analytics


In [13]:
spark.sql("select * from sales_streaming_dev.slv.regions") 

,region_id,city,processed_time
0,1,Hyderabad,2026-06-02 00:00:07.159
1,2,Delhi,2026-06-02 00:00:07.159
2,3,Mumbai,2026-06-02 00:00:07.159
3,4,Bengaluru,2026-06-02 00:00:07.159
4,5,Pune,2026-06-02 00:00:07.159
5,6,Ahmedabad,2026-06-02 00:00:07.159
